
# Assignment 9: RNN (LSTM) for Stock Price Prediction

**Name:** Abdul Shamshuddin Sheikh  
**Class:** TY CSAI - B  
**Topic:** RNN (LSTM) for Stock Price Prediction

This notebook automatically downloads stock market data using Yahoo Finance, so no manual dataset upload is required.


In [ ]:
# Install required libraries (Run once in Google Colab)
!pip install yfinance tensorflow -q


In [ ]:
# ==========================================
# Assignment 9: RNN (LSTM) for Stock Prediction
# ==========================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Input

import yfinance as yf

# ------------------------------------------
# 1. Load Dataset Automatically
# ------------------------------------------

print("Downloading Stock Dataset Automatically...")

# Download Google stock data from Yahoo Finance
data = yf.download("GOOG", start="2015-01-01", end="2024-01-01", auto_adjust=True)

# Use closing price only
df = data[['Close']]

print("\nDataset Loaded Successfully!")
print(df.head())

# ------------------------------------------
# 2. Data Preprocessing
# ------------------------------------------

# Normalize data
scaler = MinMaxScaler(feature_range=(0,1))
scaled_data = scaler.fit_transform(df)

# Create sequences
def create_dataset(data, time_step=60):
    X, y = [], []
    
    for i in range(time_step, len(data)):
        X.append(data[i-time_step:i, 0])
        y.append(data[i, 0])
        
    return np.array(X), np.array(y)

time_step = 60

X, y = create_dataset(scaled_data, time_step)

# Reshape for LSTM
X = X.reshape(X.shape[0], X.shape[1], 1)

# Train-Test Split
split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print("\nData Preprocessing Completed!")
print("Training Data Shape:", X_train.shape)
print("Testing Data Shape :", X_test.shape)

# ------------------------------------------
# 3. Build LSTM Model
# ------------------------------------------

model = Sequential([
    Input(shape=(time_step,1)),
    LSTM(50, return_sequences=True),
    LSTM(50),
    Dense(1)
])

model.compile(
    optimizer='adam',
    loss='mean_squared_error'
)

print("\nModel Summary:\n")
model.summary()

# ------------------------------------------
# 4. Train Model
# ------------------------------------------

print("\nTraining Model...\n")

history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test)
)

# ------------------------------------------
# 5. Predictions
# ------------------------------------------

print("\nMaking Predictions...\n")

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# Inverse Transform
train_pred = scaler.inverse_transform(train_pred)
test_pred = scaler.inverse_transform(test_pred)

y_train_actual = scaler.inverse_transform(y_train.reshape(-1,1))
y_test_actual = scaler.inverse_transform(y_test.reshape(-1,1))

# ------------------------------------------
# 6. Evaluation
# ------------------------------------------

train_rmse = np.sqrt(mean_squared_error(y_train_actual, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test_actual, test_pred))

print("\n==============================")
print("Train RMSE:", train_rmse)
print("Test RMSE :", test_rmse)
print("==============================")

# ------------------------------------------
# 7. Actual vs Predicted Graph
# ------------------------------------------

plt.figure(figsize=(12,6))

plt.plot(y_test_actual, label="Actual Price")
plt.plot(test_pred, label="Predicted Price")

plt.title("Stock Price Prediction using LSTM")
plt.xlabel("Time")
plt.ylabel("Stock Price")

plt.legend()

plt.show()

# ------------------------------------------
# 8. Loss Graph
# ------------------------------------------

plt.figure(figsize=(8,5))

plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')

plt.title("Loss Graph")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.legend()

plt.show()

print("\nAssignment Completed Successfully!")
